# 02 — Clean Silver Layer

Load bronze stock price data, apply cleaning transformations, compute metrics, and save to the silver layer.

**Silver columns:** `date`, `ticker`, `open`, `high`, `low`, `close`, `adj_close`, `volume`, `daily_return`, `dollar_volume`, `source`, `loaded_at`

**Flow:** bronze CSV → `clean_stock_data` → inspect → save silver CSV

**Prerequisite:** Run `01_ingest_bronze.ipynb` first so `data/processed/bronze/bronze_stock_prices.csv` exists.

## Setup

Add `src` to the Python path and import the silver transformation helper.

In [1]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path("..").resolve()
SRC_DIR = PROJECT_ROOT / "src"
BRONZE_INPUT_PATH = PROJECT_ROOT / "data" / "processed" / "bronze" / "bronze_stock_prices.csv"
SILVER_OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "silver" / "silver_stock_prices.csv"

sys.path.insert(0, str(SRC_DIR))

from transformations import clean_stock_data

print(f"Project root: {PROJECT_ROOT}")
print(f"Bronze input: {BRONZE_INPUT_PATH}")
print(f"Silver output: {SILVER_OUTPUT_PATH}")

Project root: C:\Users\bruce\OneDrive\Desktop\School\Personal Projects\financial-market-analytics-warehouse
Bronze input: C:\Users\bruce\OneDrive\Desktop\School\Personal Projects\financial-market-analytics-warehouse\data\processed\bronze\bronze_stock_prices.csv
Silver output: C:\Users\bruce\OneDrive\Desktop\School\Personal Projects\financial-market-analytics-warehouse\data\processed\silver\silver_stock_prices.csv


## Load bronze data

Read the combined bronze file created by the ingest notebook.

In [2]:
if not BRONZE_INPUT_PATH.exists():
    raise FileNotFoundError(
        f"Bronze file not found at {BRONZE_INPUT_PATH}. Run 01_ingest_bronze.ipynb first."
    )

bronze_df = pd.read_csv(BRONZE_INPUT_PATH)

print(f"Bronze rows: {len(bronze_df):,}")
print(f"Tickers: {sorted(bronze_df['ticker'].unique())}")
bronze_df.head()

Bronze rows: 210
Tickers: ['AAPL', 'MSFT']


,Date,Open,High,Low,Close,Adj Close,Volume,source_file,ticker,loaded_at
0,2024-01-02,187.149994,188.440002,183.889999,185.639999,183.562164,82488700,AAPL.csv,AAPL,2026-06-22 21:05:32.981723+00:00
1,2024-01-03,184.220001,185.880005,183.429993,184.250000,182.187744,58414500,AAPL.csv,AAPL,2026-06-22 21:05:32.981723+00:00
2,2024-01-04,182.149994,183.089996,180.880005,181.910004,179.873932,71983600,AAPL.csv,AAPL,2026-06-22 21:05:32.981723+00:00
3,2024-01-05,181.990005,182.759995,180.169998,181.179993,179.152100,62379700,AAPL.csv,AAPL,2026-06-22 21:05:32.981723+00:00
4,2024-01-08,182.089996,185.600006,181.500000,185.559998,183.483063,59144500,AAPL.csv,AAPL,2026-06-22 21:05:32.981723+00:00


## Clean and transform

`clean_stock_data` runs the full silver pipeline:

- Standardize column names
- Convert data types
- Remove duplicates
- Handle missing values
- Sort by ticker and date
- Calculate `daily_return` and `dollar_volume`

In [3]:
silver_df = clean_stock_data(bronze_df)

print(f"Silver rows: {len(silver_df):,}")
print(f"Silver columns: {list(silver_df.columns)}")
silver_df.head(10)

Silver rows: 210
Silver columns: ['date', 'ticker', 'open', 'high', 'low', 'close', 'adj_close', 'volume', 'daily_return', 'dollar_volume', 'source', 'loaded_at']


,date,ticker,open,high,low,close,adj_close,volume,daily_return,dollar_volume,source,loaded_at
0,2024-01-02,AAPL,187.149994,188.440002,183.889999,185.639999,183.562164,82488700,NaN,15313202217.652893,AAPL.csv,2026-06-22 21:05:32.981723+00:00
1,2024-01-03,AAPL,184.220001,185.880005,183.429993,184.250000,182.187744,58414500,-0.007487,10762871625.0,AAPL.csv,2026-06-22 21:05:32.981723+00:00
2,2024-01-04,AAPL,182.149994,183.089996,180.880005,181.910004,179.873932,71983600,-0.012700,13094536939.611813,AAPL.csv,2026-06-22 21:05:32.981723+00:00
3,2024-01-05,AAPL,181.990005,182.759995,180.169998,181.179993,179.152100,62379700,-0.004013,11301953589.117432,AAPL.csv,2026-06-22 21:05:32.981723+00:00
4,2024-01-08,AAPL,182.089996,185.600006,181.500000,185.559998,183.483063,59144500,0.024175,10974853275.604244,AAPL.csv,2026-06-22 21:05:32.981723+00:00
5,2024-01-09,AAPL,183.919998,185.149994,182.729996,185.139999,183.067764,42841800,-0.002263,7931730825.85144,AAPL.csv,2026-06-22 21:05:32.981723+00:00
6,2024-01-10,AAPL,184.350006,186.399994,183.919998,186.190002,184.106018,46792900,0.005671,8712370165.240479,AAPL.csv,2026-06-22 21:05:32.981723+00:00
7,2024-01-11,AAPL,186.539993,187.050003,183.619995,185.589996,183.512741,49128400,-0.003222,9117739576.086428,AAPL.csv,2026-06-22 21:05:32.981723+00:00
8,2024-01-12,AAPL,186.059998,186.740005,185.190002,185.919998,183.839050,40477800,0.001778,7525632501.882935,AAPL.csv,2026-06-22 21:05:32.981723+00:00
9,2024-01-16,AAPL,182.160004,184.259995,180.929993,183.630005,181.574692,65603000,-0.012317,12046679210.327148,AAPL.csv,2026-06-22 21:05:32.981723+00:00


## Inspect silver data

Check data types and preview the computed metrics.

In [4]:
silver_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 210 entries, 0 to 209
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   date           210 non-null    datetime64[ns]
 1   ticker         210 non-null    object        
 2   open           210 non-null    float64       
 3   high           210 non-null    float64       
 4   low            210 non-null    float64       
 5   close          210 non-null    float64       
 6   adj_close      210 non-null    float64       
 7   volume         210 non-null    Int64         
 8   daily_return   208 non-null    float64       
 9   dollar_volume  210 non-null    Float64       
 10  source         210 non-null    object        
 11  loaded_at      210 non-null    object        
dtypes: Float64(1), Int64(1), datetime64[ns](1), float64(6), object(3)
memory usage: 20.2+ KB


In [5]:
silver_df[["date", "ticker", "close", "daily_return", "volume", "dollar_volume"]].head(10)

,date,ticker,close,daily_return,volume,dollar_volume
0,2024-01-02,AAPL,185.639999,NaN,82488700,15313202217.652893
1,2024-01-03,AAPL,184.250000,-0.007487,58414500,10762871625.0
2,2024-01-04,AAPL,181.910004,-0.012700,71983600,13094536939.611813
3,2024-01-05,AAPL,181.179993,-0.004013,62379700,11301953589.117432
4,2024-01-08,AAPL,185.559998,0.024175,59144500,10974853275.604244
5,2024-01-09,AAPL,185.139999,-0.002263,42841800,7931730825.85144
6,2024-01-10,AAPL,186.190002,0.005671,46792900,8712370165.240479
7,2024-01-11,AAPL,185.589996,-0.003222,49128400,9117739576.086428
8,2024-01-12,AAPL,185.919998,0.001778,40477800,7525632501.882935
9,2024-01-16,AAPL,183.630005,-0.012317,65603000,12046679210.327148


## Save silver layer

Write the cleaned data to `data/processed/silver/`.

In [6]:
SILVER_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
silver_df.to_csv(SILVER_OUTPUT_PATH, index=False)

print(f"Saved silver data to {SILVER_OUTPUT_PATH}")

Saved silver data to C:\Users\bruce\OneDrive\Desktop\School\Personal Projects\financial-market-analytics-warehouse\data\processed\silver\silver_stock_prices.csv
